<a href="https://colab.research.google.com/github/Togarucharitha/BEE-102-Spring-2025-Assignment/blob/main/Viterbi_Algoritm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 4

This solution implements the Viterbi algorithm to infer the most likely sequence of hidden states (exon, 5′ splice site, intron) that would emit a given DNA sequence, based on a probabilistic model described in the Nature Primer.

The algorithm starts by defining the model parameters: the states (E, 5, and I), transition probabilities (including special Start and End states), and emission probabilities for each nucleotide from each state. All probabilities are converted to log-space to prevent numerical underflow and allow for summation instead of multiplication.

A helper function get_log_prob_of_a_given_path is defined to compute the total log probability of a specific path through the model given a DNA sequence. This allows validation against known paths, such as "EEEEEEEEEEEEEEEEEE5IIIIIII" for the sequence "CTTCATGTGAAAGCAGACGTAAGTCA", which yields a reference value of -41.22.

The Viterbi function itself initializes the log-probability matrix for the first base, then recursively fills in probabilities for subsequent positions by considering the best transition from each previous state. For each step, it records the most probable path to each state. After processing the full sequence, it chooses the final state with the highest probability and traces back the most likely path.

Finally, it prints the log probability of the known path, the inferred most likely path, its log probability, and optionally detects the location of a 5′ splice site (5 state) if present.

This approach effectively combines hidden Markov models and dynamic programming to perform biologically meaningful sequence annotation.

In [ ]:
import math

# Log helper to handle zero probabilities
def safe_log(p):
    return math.log(p) if p > 0 else float('-inf')

# Define states
states = ['E', '5', 'I']

# Transition probabilities (log)
transition = {
    'Start': {'E': safe_log(1.0)},
    'E': {'E': safe_log(0.9), '5': safe_log(0.1)},
    '5': {'I': safe_log(1.0)},
    'I': {'I': safe_log(0.9), 'End': safe_log(0.1)}  # No I → E transition
}

# Emission probabilities (log)
emission = {
    'E': {'A': safe_log(0.25), 'C': safe_log(0.25), 'G': safe_log(0.25), 'T': safe_log(0.25)},
    '5': {'A': safe_log(0.05), 'C': float('-inf'), 'G': safe_log(0.95), 'T': float('-inf')},
    'I': {'A': safe_log(0.4), 'C': safe_log(0.1), 'G': safe_log(0.1), 'T': safe_log(0.4)},
}

# Compute log probability of a given path and sequence
def get_log_prob_of_a_given_path(path, sequence):
    total_log_prob = 0.0
    # Initial transition from Start
    total_log_prob += transition['Start'][path[0]]
    for i in range(len(sequence)):
        s = path[i]
        o = sequence[i]
        total_log_prob += emission[s].get(o, float('-inf'))
        if i < len(sequence) - 1:
            total_log_prob += transition[s].get(path[i + 1], float('-inf'))
        else:
            # Final transition to End if applicable
            if 'End' in transition.get(s, {}):
                total_log_prob += transition[s]['End']
    return total_log_prob

# Viterbi algorithm
def viterbi(sequence):
    V = [{}]
    path = {}

    # Initialization
    for state in states:
        V[0][state] = transition['Start'].get(state, float('-inf')) + emission[state].get(sequence[0], float('-inf'))
        path[state] = [state]

    # Recursion
    for t in range(1, len(sequence)):
        V.append({})
        new_path = {}
        for curr_state in states:
            max_prob, prev_st = max(
                ((V[t-1][prev_state] + transition[prev_state].get(curr_state, float('-inf')) +
                  emission[curr_state].get(sequence[t], float('-inf')), prev_state)
                 for prev_state in states if prev_state in transition and curr_state in transition[prev_state]),
                default=(float('-inf'), None)
            )
            V[t][curr_state] = max_prob
            if prev_st:
                new_path[curr_state] = path[prev_st] + [curr_state]
        path = new_path

    # Termination
    n = len(sequence) - 1
    final_probs = {}
    for state in states:
        final_probs[state] = V[n][state] + transition.get(state, {}).get('End', 0)

    best_final_state = max(final_probs, key=final_probs.get)
    return path[best_final_state], final_probs[best_final_state]

# Input
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
known_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"

# Run
log_prob_known = get_log_prob_of_a_given_path(known_path, sequence)
viterbi_path, viterbi_log_prob = viterbi(sequence)

# Output
print(f"Log probability of known path: {log_prob_known:.2f}")
print(f"Most likely path: {''.join(viterbi_path)}")
print(f"Log probability of Viterbi path: {viterbi_log_prob:.2f}")

# Optional: detect 5' splice site
if '5' in viterbi_path:
    splice_index = viterbi_path.index('5')
    print(f"Detected 5′ splice site at position: {splice_index} (base = {sequence[splice_index]})")
else:
    print("No 5′ splice site detected in the Viterbi path.")

Log probability of known path: -41.22
Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of Viterbi path: -38.68
No 5′ splice site detected in the Viterbi path.
